In [1]:
import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
# Load Data
X_train = pd.read_csv(
    r"C:\Users\norie\Documents\Machine Learning\Projects\intel-daily-stock-price-prediction\data\processed\X_train.csv",
    index_col="Date",
    parse_dates=True
)
X_test = pd.read_csv(
    r"C:\Users\norie\Documents\Machine Learning\Projects\intel-daily-stock-price-prediction\data\processed\X_test.csv",
    index_col="Date",
    parse_dates=True
)
y_train = pd.read_csv(
    r"C:\Users\norie\Documents\Machine Learning\Projects\intel-daily-stock-price-prediction\data\processed\y_train.csv",
    index_col="Date",
    parse_dates=True
)
y_test = pd.read_csv(
    r"C:\Users\norie\Documents\Machine Learning\Projects\intel-daily-stock-price-prediction\data\processed\y_test.csv",
    index_col="Date",
    parse_dates=True
)

In [3]:
# Converting DataFrame into Series
y_train = y_train.squeeze()
y_test = y_test.squeeze()

In [4]:
# Load Scaler
scaler = joblib.load(r"C:\Users\norie\Documents\Machine Learning\Projects\intel-daily-stock-price-prediction\models\scaler.joblib")

C:\Users\norie\tf_env\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [5]:
# Scaling
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [6]:
# Linear Regression
from sklearn.linear_model import LinearRegression

lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [7]:
y_pred = lr_model.predict(X_test_scaled)

print("Coefficient:", lr_model.coef_[0])
print("Intercept:", lr_model.intercept_)
print("Predictions:", y_pred)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("R² Score:", r2_score(y_test, y_pred))

Coefficient: 0.015850262533655443
Intercept: -0.0005563653628401971
Predictions: [ 1.69491854e-03  4.27229314e-03  3.51237994e-03  3.47583318e-03
  5.25723766e-03  4.35420560e-03  3.26513947e-03  2.23583585e-03
  3.24766118e-03  6.59046683e-04  1.40117623e-03  2.00381171e-03
  1.35489647e-04  1.20807798e-03  1.50696449e-03  2.29450269e-03
  2.47074825e-03  2.74323067e-03  1.94167383e-03  1.01687046e-03
  3.75158462e-03  3.48494586e-03  1.82740487e-03  5.01653623e-03
  5.08696347e-03  5.00675791e-03  5.35775355e-03  4.87933297e-03
  2.79319916e-03 -1.87006001e-03  1.20894856e-03  2.66065347e-03
  2.02864397e-03  2.02549236e-03  3.20266897e-03  2.45494481e-03
  6.14142773e-04  2.51364148e-03  9.88271398e-06  3.19921923e-03
  1.88639964e-03  4.08192159e-04  1.14209080e-03  1.99400126e-03
 -9.31947532e-04  1.27915117e-03  7.55016110e-04 -4.32132976e-04
  1.91693779e-03  1.98973095e-03  1.75196621e-03  1.18693923e-03
  2.26184594e-03  2.50140995e-03  3.28713530e-03  3.24615244e-03
  3.16421

In [8]:
from sklearn.svm import SVR

svr_model = SVR(kernel="rbf", C=100, gamma=0.1, epsilon=0.01)
svr_model.fit(X_train_scaled, y_train)

,kernel,'rbf'
,degree,3
,gamma,0.1
,coef0,0.0
,tol,0.001
,C,100
,epsilon,0.01
,shrinking,True
,cache_size,200
,verbose,False
,max_iter,-1


In [9]:
svr_pred = svr_model.predict(X_test_scaled)

In [10]:
print("MAE:", mean_absolute_error(y_test, svr_pred))
print("MSE:", mean_squared_error(y_test, svr_pred))
print("R²:", r2_score(y_test, svr_pred))

MAE: 0.06396605295034052
MSE: 0.007538496412603363
R²: -3.11139209737338


In [11]:
import lightgbm as lgb

train_data = lgb.Dataset(X_train_scaled, label=y_train)
test_data = lgb.Dataset(X_test_scaled, label=y_test, reference=train_data)

params = {
    "objective": "regression",
    "metric": "rmse",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": -1
}

# Train model
model = lgb.train(
    params,
    train_data,
    valid_sets=[test_data],
    num_boost_round=1000
)

In [12]:
y_pred = model.predict(X_test_scaled)

# Evaluate
print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("R²:", r2_score(y_test, y_pred))

MAE: 0.034484008551275924
MSE: 0.002470464247889556
R²: -0.34735716908153735


In [14]:
single_input = pd.DataFrame([{
    "Close": 119.84,
    "MA5": 115.25,
    "MA14": 116.22,
    "Daily Return": 1.13,
    "RSI": 66.19,
    "Bollinger Band Width": 0.473,
    "High-Low Range": 4.69,
    "Relative Volume": 0.54
}])

In [15]:
prediction = lr_model.predict(single_input)

print("Predicted Target:", prediction[0])

Predicted Target: -0.32907104017664573


C:\Users\norie\tf_env\lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but LinearRegression was fitted without feature names
  warnings.warn(


In [16]:
prediction = svr_model.predict(single_input)

print("Predicted Target:", prediction[0])

Predicted Target: 0.028351878291061054


C:\Users\norie\tf_env\lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but SVR was fitted without feature names
  warnings.warn(


In [1]:
import sklearn
print(sklearn.__version__)

1.7.2
